# AIREN: OpenEnv Incident Response RL Training

This notebook trains a Qwen-1.5B model to resolve production software incidents using **Reinforcement Learning (GRPO)**. 
It connects directly to the AIREN OpenEnv Server, simulating a real environment where the system degrades every step.

**Judges:** This notebook fulfills the OpenEnv Hackathon requirement for a working end-to-end training pipeline using `unsloth` and `trl`.

### 1. Install Dependencies

In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers datasets peft torch
!pip install -q openenv-core
!pip install -q "airen-env @ git+https://github.com/LakkuAmulya-2/airen_env.git"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA Capability: {torch.cuda.get_device_capability()}")
assert torch.cuda.get_device_capability()[0] >= 8, "Please use a modern GPU (like A100 or L4) for FlashAttention support."
print("✅ All dependencies installed successfully!")

### 2. Configure Environment Connection

In [ ]:
import os
import sys

# Connect to the live AIREN OpenEnv server
ENV_URL = "https://amulyalakku-airen-env.hf.space"
os.environ["ENV_URL"] = ENV_URL

# Verify connection
import requests
try:
    resp = requests.get(f"{ENV_URL}/health", timeout=5)
    print(f"✅ Environment server is live: {resp.status_code}")
except Exception as e:
    print(f"⚠️ Warning: Could not reach environment server. Will retry during training.")
    print(f"   Error: {e}")

### 3. Load Model with Unsloth + QLoRA
We use Unsloth + QLoRA for ultra-fast, memory-efficient fine-tuning.
This allows multiple training iterations on limited compute budget.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Use SMALL model for fast iteration (winning strategy!)
# Qwen-0.5B trains 3x faster than 1.5B with similar quality
max_seq_length = 512  # Smaller for faster training
dtype = None  # Auto-detect
load_in_4bit = True  # QLoRA: 4-bit quantization
model_name = "unsloth/Qwen2.5-0.5B-Instruct"  # SMALL model for iteration

print(f"Loading {model_name}...")
print("Using QLoRA (4-bit quantization) for memory efficiency")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,  # QLoRA
)
print(f"✅ Model loaded. Memory usage: ~2GB (vs 6GB for 1.5B)")

print("Applying LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,  # Smaller rank for small model
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ LoRA applied. Trainable params: {trainable_params:,}")
print(f"   (Only {trainable_params / 500_000_000 * 100:.1f}% of model is trainable)")

### 4. TRL Environment Wrapper
AIREN adheres strictly to the `environment_factory` pattern mandated by TRL v1.0. This allows standard MDP multi-step tracking.

In [ ]:
from airen_env import AIRENEnv, AIRENAction
import os

class AIRENToolEnv:
    """TRL-compatible environment wrapper for AIREN."""
    
    def __init__(self):
        self.base_url = os.environ.get("ENV_URL", "https://amulyalakku-airen-env.hf.space")
        self._env = AIRENEnv(base_url=self.base_url)
        self._client = None
        self.reward = 0.0
        self._cumulative_reward = 0.0
        self._done = False
        self._obs = None
        self._step_count = 0

    def reset(self, **kwargs) -> str:
        """Reset environment and return initial observation."""
        try:
            if self._client is None:
                self._client = self._env.sync().__enter__()
            
            self.reward = 0.0
            self._cumulative_reward = 0.0
            self._done = False
            self._step_count = 0
            
            result = self._client.reset()
            self._obs = result.observation
            return self._format_obs()
        except Exception as e:
            print(f"Reset error: {e}")
            raise

    def run_diagnostic(self, target: str) -> str:
        return self._step("run_diagnostic", target, f"Running diagnostic on {target}")
    
    def inspect_logs(self, target: str) -> str:
        return self._step("inspect_logs", target, f"Inspecting logs for {target}")
    
    def inspect_metrics(self, target: str) -> str:
        return self._step("inspect_metrics", target, f"Checking metrics for {target}")
    
    def apply_fix(self, target: str) -> str:
        return self._step("apply_fix", target, f"Applying fix to {target}")
    
    def restart_service(self, target: str) -> str:
        return self._step("restart_service", target, f"Restarting {target}")
    
    def rollback_deployment(self, target: str) -> str:
        return self._step("rollback_deployment", target, f"Rolling back {target}")
    
    def scale_service(self, target: str) -> str:
        return self._step("scale_service", target, f"Scaling up {target}")
    
    def acknowledge_incident(self) -> str:
        return self._step("acknowledge_incident", "system", "Acknowledging incident")
        
    def _step(self, action_type: str, target: str, reasoning: str) -> str:
        """Execute action and return observation."""
        if self._done:
            return "[EPISODE ALREADY ENDED]"
        
        try:
            result = self._client.step(AIRENAction(
                action_type=action_type,
                target=target,
                reasoning=reasoning
            ))
            
            obs = result.observation
            self._cumulative_reward += (result.reward or 0.0)
            self.reward = self._cumulative_reward
            self._done = obs.done
            self._obs = obs
            self._step_count += 1
            
            if self._done:
                return f"[COMPLETE] Health: {obs.system_health:.0%} | Resolved: {obs.incident_resolved} | Reward: {self._cumulative_reward:.3f}"
            
            latest_log = obs.logs[-1] if obs.logs else "No logs"
            return f"Health: {obs.system_health:.0%} | Step: {self._step_count} | Log: {latest_log}"
        except Exception as e:
            print(f"Step error: {e}")
            self._done = True
            return f"[ERROR] {str(e)}"

    def _format_obs(self) -> str:
        """Format observation for display."""
        obs = self._obs
        services_str = "\n".join([f"  {n}: {s['status']} (err={s['error_rate']:.0%})" 
                                   for n, s in obs.services.items()])
        return f"INCIDENT: {obs.incident_type}\nHealth: {obs.system_health:.0%}\nSERVICES:\n{services_str}"

def reward_func(environments, **kwargs) -> list:
    """Extract rewards from environments for GRPO trainer."""
    return [env._cumulative_reward for env in environments]

print("✅ Environment wrapper ready!")

### 5. Configure GRPO Trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import json

system_prompt = (
    "You are an expert SRE. Diagnose before fixing. System degrades every step.\n"
    "Available tools: run_diagnostic, inspect_logs, inspect_metrics, apply_fix, "
    "restart_service, rollback_deployment, scale_service, acknowledge_incident"
)

# Create dataset with system prompts
dataset = Dataset.from_dict({
    "prompt": [[{"role": "user", "content": system_prompt}]] * 32 
})

print(f"Dataset size: {len(dataset)} samples")

# WINNING STRATEGY: Multiple small training runs instead of one big run
# This allows iteration and refinement
config = GRPOConfig(
    output_dir="./airen-grpo-output",
    learning_rate=5e-5,  # Slightly higher for small model
    per_device_train_batch_size=4,  # Larger batch for small model
    gradient_accumulation_steps=2,  # Smaller accumulation
    max_completion_length=128,  # Smaller for faster generation
    num_generations=4,
    num_train_epochs=2,  # Multiple epochs for iteration
    logging_steps=1,
    save_steps=5,
    eval_strategy="no",
    use_vllm=False,  # vLLM not needed for small model
    seed=42,
)

print(f"✅ GRPO config ready (optimized for iteration)")
print(f"   Model: Qwen-0.5B (small, fast)")
print(f"   Batch size: {config.per_device_train_batch_size}")
print(f"   Epochs: {config.num_train_epochs}")
print(f"   Expected time: ~5-10 min per run")
print(f"   Strategy: Multiple iterations > single large run")

### 6. Train with GRPO (Multiple Iterations)
WINNING STRATEGY: Run multiple small training iterations instead of one large run.
This allows you to:
- See results faster (5-10 min per run)
- Iterate on reward signals
- Adjust hyperparameters
- Show multiple training curves
- Demonstrate learning progression

Expected time: ~5-10 minutes per iteration on Colab GPU

In [ ]:
print("Starting GRPO training (Iteration 1)...")
print(f"Total training steps: {len(dataset) * config.num_train_epochs // (config.per_device_train_batch_size * config.gradient_accumulation_steps)}")
print()

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=config,
    environment_factory=AIRENToolEnv,
)

# Train
train_result = trainer.train()

print("\n✅ Training Iteration 1 complete!")
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"\nTo run more iterations:")
print("1. Modify reward_func or environment parameters")
print("2. Run this cell again")
print("3. Compare results across iterations")

### 7. Extract and Plot Learning Curves
Generate evidence plots for judges showing training progress.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json

# Extract training history
log_history = trainer.state.log_history

# Parse loss and reward curves
steps = []
losses = []
rewards = []

for entry in log_history:
    if 'loss' in entry:
        steps.append(entry.get('step', len(steps)))
        losses.append(entry['loss'])
    if 'reward' in entry:
        rewards.append(entry['reward'])

print(f"Extracted {len(steps)} training steps")
print(f"Loss range: {min(losses):.4f} to {max(losses):.4f}")
if rewards:
    print(f"Reward range: {min(rewards):.4f} to {max(rewards):.4f}")

# Plot 1: Loss Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(steps, losses, 'b-', linewidth=2, label='Training Loss')
ax1.set_xlabel('Training Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('GRPO Training Loss over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Reward Curve (if available)
if rewards:
    ax2.plot(range(len(rewards)), rewards, 'g-', linewidth=2, label='Episode Reward')
    ax2.set_xlabel('Episode', fontsize=12)
    ax2.set_ylabel('Cumulative Reward', fontsize=12)
    ax2.set_title('Average Reward During Training', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'Reward data not yet available\n(will populate after first episodes)', 
             ha='center', va='center', fontsize=12, transform=ax2.transAxes)
    ax2.set_title('Average Reward During Training', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('airen_training_curves.png', dpi=150, bbox_inches='tight')
print("\n✅ Saved: airen_training_curves.png")
plt.show()

### 8. Save Training Metrics
Export metrics for README and documentation.

In [ ]:
# Save training metrics
metrics = {
    "model": model_name,
    "training_steps": len(steps),
    "final_loss": float(losses[-1]) if losses else None,
    "min_loss": float(min(losses)) if losses else None,
    "max_loss": float(max(losses)) if losses else None,
    "avg_reward": float(np.mean(rewards)) if rewards else None,
    "max_reward": float(max(rewards)) if rewards else None,
    "learning_rate": config.learning_rate,
    "batch_size": config.per_device_train_batch_size,
}

with open('airen_training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✅ Training Metrics:")
for key, value in metrics.items():
    print(f"   {key}: {value}")

print("\n✅ Saved: airen_training_metrics.json")

### 9. Save Trained Model
Save the trained model for inference and deployment.

In [ ]:
# Save model
model.save_pretrained("airen-qwen-grpo-trained")
tokenizer.save_pretrained("airen-qwen-grpo-trained")

print("✅ Model saved to: airen-qwen-grpo-trained/")
print("\nTo push to HuggingFace Hub:")
print("  model.push_to_hub('your-username/airen-qwen-grpo-trained')")
print("  tokenizer.push_to_hub('your-username/airen-qwen-grpo-trained')")

### 10. Run Inference Demo
Test the trained model on a sample incident.

In [ ]:
# Quick inference test
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=100)

prompt = "You are an expert SRE. A database is overloaded. What do you do?"
result = pipe(prompt)

print("Sample Inference:")
print(f"Prompt: {prompt}")
print(f"Response: {result[0]['generated_text']}")